In [25]:
!pip install transformers datasets evaluate accelerate torch scikit-learn gradio -q

In [26]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score

In [27]:
from datasets import load_dataset

dataset  = load_dataset("sh0416/ag_news")

In [28]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [29]:
print(dataset)

print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})
['label', 'title', 'description']


In [30]:
def tokenize_function(example):

    combined_text = [
        title + " " + description
        for title, description in zip(
            example["title"],
            example["description"]
        )
    ]

    return tokenizer(
        combined_text,
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [31]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["title", "description"]
)

tokenized_dataset.set_format("torch")

In [32]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
def compute_metrics(pred):

    labels = pred.label_ids
    predictions = pred.predictions.argmax(-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    f1 = f1_score(
        labels,
        predictions,
        average='weighted'
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [34]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True
)


In [35]:
!pip uninstall -y torch torchvision torchaudio transformers datasets accelerate

!pip install -q \
torch \
torchvision \
transformers==4.52.4 \
datasets==3.5.0 \
accelerate==1.6.0

Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: datasets 3.5.0
Uninstalling datasets-3.5.0:
  Successfully uninstalled datasets-3.5.0
Found existing installation: accelerate 1.6.0
Uninstalling accelerate-1.6.0:
  Successfully uninstalled accelerate-1.6.0


In [36]:
!pip uninstall -y torchvision
!pip install torchvision -q

Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0


In [37]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["test"],

    compute_metrics=compute_metrics
)

trainer.train()

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
model.save_pretrained(
    "./saved_model"
)

tokenizer.save_pretrained(
    "./saved_model"
)

In [ ]:
import gradio as gr
import torch

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

model_path="./saved_model"

tokenizer= BertTokenizer.from_pretrained(
    model_path
)

model= BertForSequenceClassification.from_pretrained(
    model_path
)

labels = {
    0:"World",
    1:"Sports",
    2:"Business",
    3:"Sci/Tech"
}


def classify_news(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs=model(**inputs)

        prediction=torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    return labels[prediction]


interface=gr.Interface(
    fn=classify_news,
    inputs="textbox",
    outputs="text",
    title="News Topic Classifier using BERT",
    description="Enter a news headline"
)

interface.launch()

In [ ]:
interface.launch(share=True)